# XLM-RoBERTa Fine-tuning for Multilingual Sentiment

This notebook consolidates the cleaned datasets across languages and fine-tunes `xlm-roberta-base` for sentiment classification with class-imbalance handling.

In [1]:
import os
import math
from pathlib import Path
import time
from collections import Counter, defaultdict
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score, classification_report
import torch
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from transformers import (
    XLMRobertaTokenizerFast,
    XLMRobertaForSequenceClassification,
    DataCollatorWithPadding,
    get_linear_schedule_with_warmup,
    get_cosine_schedule_with_warmup,
 )
from tqdm.auto import tqdm

/home/nibiru/miniconda3/envs/dl-env/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
BASE_DIR = Path("/home/nibiru/Documents/projects/Text_try2")
CLEANED_DIR = BASE_DIR / "cleaned_data"
LANG_FILES = {
    "english": CLEANED_DIR / "english_cleaned.csv",
    "kannada": CLEANED_DIR / "kannada_cleaned.csv",
    "malayalam": CLEANED_DIR / "malayalam_cleaned.csv",
    "tamil": CLEANED_DIR / "tamil_cleaned.csv",
}

records = []
for lang, path in LANG_FILES.items():
    if not path.is_file():
        raise FileNotFoundError(f"Missing cleaned data for {lang}: {path}")
    df = pd.read_csv(path)
    if {"sentence", "label"} - set(df.columns):
        raise ValueError(f"File {path} missing required columns 'sentence' and 'label'")
    df = df.dropna(subset=["sentence", "label"]).copy()
    df["sentence"] = df["sentence"].astype(str).str.strip()
    df = df[df["sentence"] != ""]
    df["language"] = lang
    records.append(df[["sentence", "label", "language"]])

master_df = pd.concat(records, ignore_index=True)
master_df = master_df.rename(columns={"sentence": "text"})
print(master_df.head())
print("\nSamples per language:")
print(master_df.groupby("language").size())
print("\nLabel distribution:")
print(master_df["label"].value_counts())

                                                text     label language
0                     anyone know what movie this is   Neutral  english
1  the fact theyre holding each other back while ...  Positive  english
2                         waiting next video will be   Neutral  english
3  thanks for the great video i dont understand w...   Neutral  english
4  good person helping good people this is how it...  Positive  english

Samples per language:
language
english      1031236
kannada        24510
malayalam      71966
tamil         162008
dtype: int64

Label distribution:
label
Neutral     485619
Positive    415883
Negative    388218
Name: count, dtype: int64


In [3]:
LABEL_COLUMN = "label"
TEXT_COLUMN = "text"
LANG_COLUMN = "language"

label_order = sorted(master_df[LABEL_COLUMN].unique())
label2id = {label: idx for idx, label in enumerate(label_order)}
id2label = {idx: label for label, idx in label2id.items()}
master_df["label_id"] = master_df[LABEL_COLUMN].map(label2id)

print("Label mapping:")
for label, idx in label2id.items():
    print(f"  {label} -> {idx}")

MAX_TOTAL_SAMPLES = 400_000  # set to None to use the full dataset
SEED = 42
TEST_SIZE = 0.15

if MAX_TOTAL_SAMPLES is not None and len(master_df) > MAX_TOTAL_SAMPLES:
    master_df, _ = train_test_split(
        master_df,
        train_size=MAX_TOTAL_SAMPLES,
        stratify=master_df["label_id"],
        random_state=SEED,
    )
    master_df = master_df.reset_index(drop=True)
    print(f"\nDown-sampled dataset to {len(master_df):,} rows for manageable fine-tuning.")

train_df, val_df = train_test_split(
    master_df,
    test_size=TEST_SIZE,
    stratify=master_df["label_id"],
    random_state=SEED,
 )

train_df = train_df.reset_index(drop=True)
val_df = val_df.reset_index(drop=True)
print(f"\nTrain samples: {len(train_df):,}")
print(f"Validation samples: {len(val_df):,}")

train_counts = Counter(train_df["label_id"].tolist())
val_counts = Counter(val_df["label_id"].tolist())
class_weights = np.array([
    len(train_df) / (len(train_counts) * train_counts[idx]) for idx in range(len(label2id))
], dtype=np.float32)
train_df["sample_weight"] = train_df["label_id"].map(lambda idx: class_weights[idx])

print("\nTraining label counts:")
for idx, count in sorted(train_counts.items()):
    print(f"  class {idx} ({id2label[idx]}): {count:,}")
print("Class weights:", class_weights)

print("\nValidation label counts:")
for idx, count in sorted(val_counts.items()):
    print(f"  class {idx} ({id2label[idx]}): {count:,}")

print("\nPer-language distribution (train / val):")
for lang in sorted(master_df[LANG_COLUMN].unique()):
    train_lang = (train_df[LANG_COLUMN] == lang).sum()
    val_lang = (val_df[LANG_COLUMN] == lang).sum()
    print(f"  {lang:<10} -> train={train_lang:,} | val={val_lang:,}")

Label mapping:
  Negative -> 0
  Neutral -> 1
  Positive -> 2

Down-sampled dataset to 400,000 rows for manageable fine-tuning.

Train samples: 340,000
Validation samples: 60,000

Training label counts:
  class 0 (Negative): 102,343
  class 1 (Neutral): 128,020
  class 2 (Positive): 109,637
Class weights: [1.1073873  0.88527834 1.0337143 ]

Validation label counts:
  class 0 (Negative): 18,061
  class 1 (Neutral): 22,592
  class 2 (Positive): 19,347

Per-language distribution (train / val):
  english    -> train=271,850 | val=48,175
  kannada    -> train=6,604 | val=1,111
  malayalam  -> train=18,790 | val=3,330
  tamil      -> train=42,756 | val=7,384

Down-sampled dataset to 400,000 rows for manageable fine-tuning.

Train samples: 340,000
Validation samples: 60,000

Training label counts:
  class 0 (Negative): 102,343
  class 1 (Neutral): 128,020
  class 2 (Positive): 109,637
Class weights: [1.1073873  0.88527834 1.0337143 ]

Validation label counts:
  class 0 (Negative): 18,061
  cl

In [4]:
MODEL_NAME = "xlm-roberta-base"
MAX_LENGTH = 160
TRAIN_BATCH_SIZE = 32
EVAL_BATCH_SIZE = 64
GRADIENT_ACCUMULATION_STEPS = 2
NUM_EPOCHS = 5
LEARNING_RATE = 1.5e-5
MIN_LEARNING_RATE = 5e-6
WARMUP_RATIO = 0.06
WEIGHT_DECAY = 0.01
MAX_GRAD_NORM = 1.0
EARLY_STOPPING_PATIENCE = 2
USE_WEIGHTED_SAMPLER = True
USE_TORCH_COMPILE = torch.cuda.is_available() and torch.__version__.startswith("2.")
LR_SCHEDULER_TYPE = "cosine"  # options: 'linear', 'cosine'

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
    torch.backends.cudnn.benchmark = True
    torch.backends.cuda.matmul.allow_tf32 = True
if hasattr(torch, "set_float32_matmul_precision"):
    torch.set_float32_matmul_precision("high")

tokenizer = XLMRobertaTokenizerFast.from_pretrained(MODEL_NAME)

class SentimentDataset(Dataset):
    def __init__(self, df, tokenizer, max_length):
        self.texts = df[TEXT_COLUMN].tolist()
        self.labels = df["label_id"].tolist()
        self.languages = df[LANG_COLUMN].tolist()
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        text = self.texts[idx]
        encoding = self.tokenizer(
            text,
            truncation=True,
            max_length=self.max_length,
            add_special_tokens=True,
        )
        encoding["labels"] = self.labels[idx]
        encoding["language"] = self.languages[idx]
        return encoding

train_dataset = SentimentDataset(train_df, tokenizer, MAX_LENGTH)
val_dataset = SentimentDataset(val_df, tokenizer, MAX_LENGTH)

base_collator = DataCollatorWithPadding(tokenizer=tokenizer, padding="longest", return_tensors="pt")

def collate_with_language(batch):
    languages = [item.pop("language") for item in batch]
    encoded_batch = base_collator(batch)
    encoded_batch["language"] = languages
    return encoded_batch

num_workers = max(1, (os.cpu_count() or 1) // 2)
if USE_WEIGHTED_SAMPLER:
    sample_weights = torch.as_tensor(train_df["sample_weight"].values, dtype=torch.double)
    train_sampler = WeightedRandomSampler(
        weights=sample_weights,
        num_samples=len(sample_weights),
        replacement=True,
        generator=torch.Generator().manual_seed(SEED),
    )
else:
    train_sampler = None

train_loader = DataLoader(
    train_dataset,
    batch_size=TRAIN_BATCH_SIZE,
    shuffle=train_sampler is None,
    sampler=train_sampler,
    collate_fn=collate_with_language,
    num_workers=num_workers,
    pin_memory=device.type == "cuda",
    persistent_workers=num_workers > 0,
    drop_last=False,
 )
val_loader = DataLoader(
    val_dataset,
    batch_size=EVAL_BATCH_SIZE,
    shuffle=False,
    collate_fn=collate_with_language,
    num_workers=num_workers,
    pin_memory=device.type == "cuda",
    persistent_workers=num_workers > 0,
 )

class_weights_tensor = torch.from_numpy(class_weights).to(device)
print("Train batches:", len(train_loader), "| Val batches:", len(val_loader))

Using device: cuda
Train batches: 10625 | Val batches: 938
Train batches: 10625 | Val batches: 938


In [5]:
def evaluate(model, data_loader, loss_fn, device):
    model.eval()
    total_loss = 0.0
    total_examples = 0
    preds_list = []
    labels_list = []
    languages_list = []
    with torch.no_grad():
        for batch in data_loader:
            languages = batch.pop("language")
            labels = batch.pop("labels").to(device)
            inputs = {k: v.to(device) for k, v in batch.items()}
            outputs = model(**inputs)
            loss = loss_fn(outputs.logits, labels)
            total_loss += loss.item() * labels.size(0)
            total_examples += labels.size(0)
            preds = outputs.logits.argmax(dim=1)
            preds_list.append(preds.cpu())
            labels_list.append(labels.cpu())
            languages_list.extend(languages)
    avg_loss = total_loss / total_examples if total_examples else 0.0
    preds = torch.cat(preds_list).numpy() if preds_list else np.array([])
    labels = torch.cat(labels_list).numpy() if labels_list else np.array([])
    macro_f1 = f1_score(labels, preds, average="macro") if labels.size else 0.0
    per_class_f1 = f1_score(labels, preds, average=None) if labels.size else np.zeros(len(id2label))
    report = classification_report(
        labels,
        preds,
        target_names=[id2label[i] for i in range(len(id2label))],
        digits=4,
        zero_division=0,
    ) if labels.size else "No validation samples evaluated."
    per_language_scores = {}
    if languages_list:
        lang_groups = defaultdict(lambda: {"labels": [], "preds": []})
        for gold, pred, lang in zip(labels, preds, languages_list):
            lang_groups[lang]["labels"].append(gold)
            lang_groups[lang]["preds"].append(pred)
        for lang, values in lang_groups.items():
            lang_labels = np.array(values["labels"])
            lang_preds = np.array(values["preds"])
            per_language_scores[lang] = f1_score(lang_labels, lang_preds, average="macro", zero_division=0)
        per_language_scores = dict(sorted(per_language_scores.items(), key=lambda x: x[0]))
    return avg_loss, macro_f1, per_class_f1, per_language_scores, report, preds, labels, languages_list

In [6]:
model = XLMRobertaForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=len(label2id),
    id2label=id2label,
    label2id=label2id,
 ).to(device)
if USE_TORCH_COMPILE:
    try:
        model = torch.compile(model)
        print("Enabled torch.compile for faster training")
    except Exception as compile_err:
        print(f"torch.compile unavailable ({compile_err}); continuing without it")

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=LEARNING_RATE,
    betas=(0.9, 0.98),
    eps=1e-6,
    weight_decay=WEIGHT_DECAY,
 )
updates_per_epoch = math.ceil(len(train_loader) / GRADIENT_ACCUMULATION_STEPS)
total_training_steps = updates_per_epoch * NUM_EPOCHS
warmup_steps = max(1, int(total_training_steps * WARMUP_RATIO))
print(f"Total training steps: {total_training_steps} | Warmup steps: {warmup_steps}")

if LR_SCHEDULER_TYPE == "cosine":
    scheduler = get_cosine_schedule_with_warmup(
        optimizer,
        num_warmup_steps=warmup_steps,
        num_training_steps=total_training_steps,
    )
else:
    scheduler = get_linear_schedule_with_warmup(
        optimizer,
        num_warmup_steps=warmup_steps,
        num_training_steps=total_training_steps,
    )

loss_fn = torch.nn.CrossEntropyLoss(weight=class_weights_tensor)
scaler = torch.cuda.amp.GradScaler(enabled=device.type == "cuda")

class EarlyStopping:
    def __init__(self, patience=2, min_delta=1e-4, mode="max"):
        self.patience = patience
        self.min_delta = min_delta
        self.mode = mode
        self.best = -float("inf") if mode == "max" else float("inf")
        self.num_bad_epochs = 0

    def step(self, metric):
        improvement = (metric - self.best) if self.mode == "max" else (self.best - metric)
        if improvement > self.min_delta:
            self.best = metric
            self.num_bad_epochs = 0
        else:
            self.num_bad_epochs += 1
        return self.num_bad_epochs > self.patience

early_stopper = EarlyStopping(patience=EARLY_STOPPING_PATIENCE)

history = []
best_state = None
best_macro_f1 = -1.0

for epoch in range(1, NUM_EPOCHS + 1):
    model.train()
    optimizer.zero_grad(set_to_none=True)
    running_loss = 0.0
    running_correct = 0
    running_examples = 0

    progress_bar = tqdm(train_loader, desc=f"Epoch {epoch}/{NUM_EPOCHS}", leave=False)
    for step, batch in enumerate(progress_bar, start=1):
        _ = batch.pop("language", None)  # language not needed during forward pass
        labels = batch.pop("labels").to(device)
        inputs = {k: v.to(device) for k, v in batch.items()}

        with torch.cuda.amp.autocast(enabled=device.type == "cuda"):
            outputs = model(**inputs)
            loss = loss_fn(outputs.logits, labels)
            loss = loss / GRADIENT_ACCUMULATION_STEPS

        scaler.scale(loss).backward()
        running_loss += loss.item() * GRADIENT_ACCUMULATION_STEPS
        preds = outputs.logits.argmax(dim=1)
        running_correct += (preds == labels).sum().item()
        running_examples += labels.size(0)

        if step % GRADIENT_ACCUMULATION_STEPS == 0 or step == len(train_loader):
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), MAX_GRAD_NORM)
            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad(set_to_none=True)
            scheduler.step()

        if step % 100 == 0 or step == len(train_loader):
            current_lr = scheduler.get_last_lr()[0]
            progress_bar.set_postfix({
                "loss": f"{running_loss / step:.4f}",
                "lr": f"{current_lr:.2e}",
            })

    train_loss = running_loss / len(train_loader)
    train_acc = running_correct / running_examples if running_examples else 0.0
    print(f"Epoch {epoch} completed | train_loss={train_loss:.4f} train_acc={train_acc*100:.2f}%")

    val_results = evaluate(model, val_loader, loss_fn, device)
    val_loss, val_macro_f1, val_per_class_f1, val_lang_scores, val_report, _, _, _ = val_results
    print(f"Validation loss={val_loss:.4f} | Macro F1={val_macro_f1:.4f}")
    for idx, score in enumerate(val_per_class_f1):
        print(f"  Class {idx} ({id2label[idx]}): F1={score:.4f}")
    if val_lang_scores:
        print("Per-language macro F1:")
        for lang, score in val_lang_scores.items():
            print(f"  {lang}: {score:.4f}")

    history.append({
        "epoch": epoch,
        "train_loss": train_loss,
        "train_acc": train_acc,
        "val_loss": val_loss,
        "val_macro_f1": val_macro_f1,
        "val_per_class_f1": val_per_class_f1.tolist(),
        "val_lang_scores": val_lang_scores,
    })

    if val_macro_f1 > best_macro_f1:
        best_macro_f1 = val_macro_f1
        best_state = {
            "model_state": model.state_dict(),
            "optimizer_state": optimizer.state_dict(),
            "epoch": epoch,
            "val_loss": val_loss,
            "val_macro_f1": val_macro_f1,
            "val_report": val_report,
            "val_lang_scores": val_lang_scores,
            "history": history.copy(),
        }
        torch.save(best_state, BASE_DIR / "xlmr_finetune_best.pt")
        print(f"Saved new best model checkpoint (Macro F1={val_macro_f1:.4f})")

    if early_stopper.step(val_macro_f1):
        print(
            f"Early stopping triggered after epoch {epoch}; no macro F1 improvement "
            f"for {EARLY_STOPPING_PATIENCE} consecutive checks.",
        )
        break

if best_state is not None:
    model.load_state_dict(best_state["model_state"])
    history = best_state.get("history", history)
    print(
        f"Loaded best model from epoch {best_state['epoch']} "
        f"(val_macro_f1={best_state['val_macro_f1']:.4f}, val_loss={best_state['val_loss']:.4f})"
    )
else:
    print("Warning: training did not produce a valid checkpoint.")

Some weights of XLMRobertaForSequenceClassification were not initialized from the model checkpoint at xlm-roberta-base and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
/tmp/ipykernel_9805/282302800.py:40: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler(enabled=device.type == "cuda")
/tmp/ipykernel_9805/282302800.py:40: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler(enabled=device.type == "cuda")


Enabled torch.compile for faster training
Total training steps: 26565 | Warmup steps: 1593


Epoch 1/5:   0%|          | 0/10625 [00:00<?, ?it/s]/tmp/ipykernel_9805/282302800.py:78: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=device.type == "cuda"):
/tmp/ipykernel_9805/282302800.py:78: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=device.type == "cuda"):
Epoch 1/5:   0%|          | 1/10625 [00:19<57:26:49, 19.47s/it]/tmp/ipykernel_9805/282302800.py:78: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=device.type == "cuda"):
/tmp/ipykernel_9805/282302800.py:78: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=device.type == "c

Epoch 1 completed | train_loss=0.7358 train_acc=65.98%


Validation loss=0.6795 | Macro F1=0.6994
  Class 0 (Negative): F1=0.7209
  Class 1 (Neutral): F1=0.6636
  Class 2 (Positive): F1=0.7136
Per-language macro F1:
  english: 0.7279
  kannada: 0.4845
  malayalam: 0.4153
  tamil: 0.4818
Saved new best model checkpoint (Macro F1=0.6994)
Saved new best model checkpoint (Macro F1=0.6994)


Epoch 2/5:   0%|          | 0/10625 [00:00<?, ?it/s]/tmp/ipykernel_9805/282302800.py:78: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=device.type == "cuda"):
/tmp/ipykernel_9805/282302800.py:78: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=device.type == "cuda"):
                                                                                          

... (3 outputs left out)

Epoch 4/5:   0%|          | 0/10625 [00:00<?, ?it/s]/tmp/ipykernel_9805/282302800.py:78: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=device.type == "cuda"):
/tmp/ipykernel_9805/282302800.py:78: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=device.type == "cuda"):
                                                                                          

Epoch 4 completed | train_loss=0.5096 train_acc=77.39%
Validation loss=0.6908 | Macro F1=0.7131
  Class 0 (Negative): F1=0.7438
  Class 1 (Neutral): F1=0.6649
  Class 2 (Positive): F1=0.7305
Per-language macro F1:
  english: 0.7440
  kannada: 0.5500
  malayalam: 0.5129
  tamil: 0.5309
Validation loss=0.6908 | Macro F1=0.7131
  Class 0 (Negative): F1=0.7438
  Class 1 (Neutral): F1=0.6649
  Class 2 (Positive): F1=0.7305
Per-language macro F1:
  english: 0.7440
  kannada: 0.5500
  malayalam: 0.5129
  tamil: 0.5309
Saved new best model checkpoint (Macro F1=0.7131)
Saved new best model checkpoint (Macro F1=0.7131)


Epoch 5/5:   0%|          | 0/10625 [00:00<?, ?it/s]/tmp/ipykernel_9805/282302800.py:78: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=device.type == "cuda"):
/tmp/ipykernel_9805/282302800.py:78: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=device.type == "cuda"):


Epoch 5 completed | train_loss=0.4924 train_acc=78.04%
Validation loss=0.6948 | Macro F1=0.7112
  Class 0 (Negative): F1=0.7441
  Class 1 (Neutral): F1=0.6577
  Class 2 (Positive): F1=0.7317
Per-language macro F1:
  english: 0.7437
  kannada: 0.5530
  malayalam: 0.5286
  tamil: 0.5281
Loaded best model from epoch 4 (val_macro_f1=0.7131, val_loss=0.6908)
Validation loss=0.6948 | Macro F1=0.7112
  Class 0 (Negative): F1=0.7441
  Class 1 (Neutral): F1=0.6577
  Class 2 (Positive): F1=0.7317
Per-language macro F1:
  english: 0.7437
  kannada: 0.5530
  malayalam: 0.5286
  tamil: 0.5281
Loaded best model from epoch 4 (val_macro_f1=0.7131, val_loss=0.6908)


In [7]:
print("Best validation classification report:\n")
print(best_state["val_report"])
lang_scores = best_state.get("val_lang_scores", {})
if lang_scores:
    print("\nBest +per-language macro F1:")
    for lang, score in lang_scores.items():
        print(f"  {lang}: {score:.4f}")

history_df = pd.DataFrame(history) if history else pd.DataFrame()
if not history_df.empty:
    display(history_df[["epoch", "train_loss", "val_loss", "val_macro_f1"]])

Best validation classification report:

              precision    recall  f1-score   support

    Negative     0.6983    0.7956    0.7438     18061
     Neutral     0.7109    0.6246    0.6649     22592
    Positive     0.7263    0.7348    0.7305     19347

    accuracy                         0.7116     60000
   macro avg     0.7118    0.7183    0.7131     60000
weighted avg     0.7121    0.7116    0.7098     60000


Best +per-language macro F1:
  english: 0.7440
  kannada: 0.5500
  malayalam: 0.5129
  tamil: 0.5309


,epoch,train_loss,val_loss,val_macro_f1
0,1,0.735847,0.679459,0.699393
1,2,0.609136,0.678266,0.702528
2,3,0.553311,0.677026,0.703921
3,4,0.509589,0.690822,0.713085


In [22]:
from torch.nn.functional import softmax

checkpoint_path = BASE_DIR / "xlmr_finetune_best.pt"
if "best_state" in globals() and best_state is not None:
    print(
        f"Using best_state from epoch {best_state['epoch']} "
        f"(Macro F1={best_state['val_macro_f1']:.4f})"
    )
elif checkpoint_path.is_file():
    checkpoint = torch.load(checkpoint_path, map_location=device)
    if "model_state" in checkpoint:
        model.load_state_dict(checkpoint["model_state"])
        print("Loaded model weights from checkpoint.")
    else:
        print("Checkpoint found but missing 'model_state'; using current weights.")
else:
    print("Checkpoint not found; using current in-memory model weights.")

model.eval()

def predict_sentiment(text, max_length=MAX_LENGTH):
    if not isinstance(text, str) or not text.strip():
        raise ValueError("Input text must be a non-empty string.")
    encoded = tokenizer(
        text.strip(),
        return_tensors="pt",
        truncation=True,
        max_length=max_length,
    )
    encoded = {k: v.to(device) for k, v in encoded.items()}
    with torch.no_grad():
        logits = model(**encoded).logits
        probs = softmax(logits, dim=-1).squeeze(0).cpu()
    pred_idx = int(probs.argmax())
    return {
        "text": text,
        "predicted_label": id2label[pred_idx],
        "probabilities": {id2label[i]: float(probs[i]) for i in range(len(id2label))}
    }

example_text = "எனக்கு இப்போது கோபம் வருகிறது."
prediction = predict_sentiment(example_text)
print("Input:", prediction["text"])
print("Predicted label:", prediction["predicted_label"])
print("Probabilities:")
for label, prob in prediction["probabilities"].items():
    print(f"  {label}: {prob:.4f}")

Using best_state from epoch 4 (Macro F1=0.7131)
Input: எனக்கு இப்போது கோபம் வருகிறது.
Predicted label: Negative
Probabilities:
  Negative: 0.9185
  Neutral: 0.0677
  Positive: 0.0138


In [12]:
pip install matplotlib seaborn

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 0.0/8.7 MB ? eta -:--:--Downloading matplotlib-3.10.6-cp312-cp312-manylinux2014_x86_64.manylinux_2_17_x86_64.whl (8.7 MB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.7/8.7 MB 2.7 MB/s  0:00:03 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.7/8.7 MB 2.7 MB/s  0:00:03
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 0.0/4.9 MB ? eta -:--:--Downloading fonttools-4.60.0-cp312-cp312-manylinux1_x86_64.manylinux2014_x86_64.manylinux_2_17_x86_64.manylinux_2_5_x86_64.whl (4.9 MB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 1.7 MB/s  0:00:02 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 1.7 MB/s  0:00:02
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 0.0/1.5 MB ? eta -:--:--Downloading kiwisolver-1.4.9-cp312-cp312-manylinux2014_x86_64.manylinux_2_17_x86_64.whl (1.5 MB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 931.2 kB/s  0:00:01ta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 931.2 kB/

In [11]:
try:
    import seaborn as sns
except ImportError:
    sns = None
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix

val_summary = evaluate(model, val_loader, loss_fn, device)
_, final_macro_f1, final_class_f1, final_lang_scores, final_report, final_preds, final_labels, final_languages = val_summary
print(f"Final validation macro F1 (best checkpoint): {final_macro_f1:.4f}")
print(final_report)
if final_lang_scores:
    print("Per-language macro F1:")
    for lang, score in final_lang_scores.items():
        print(f"  {lang}: {score:.4f}")

cm = confusion_matrix(final_labels, final_preds)
fig, ax = plt.subplots(figsize=(6, 5))
if sns is not None:
    sns.heatmap(
        cm,
        annot=True,
        fmt="d",
        cmap="Blues",
        xticklabels=[id2label[i] for i in range(len(id2label))],
        yticklabels=[id2label[i] for i in range(len(id2label))],
        ax=ax,
    )
else:
    im = ax.imshow(cm, interpolation="nearest", cmap="Blues")
    plt.colorbar(im, ax=ax)
    for i in range(cm.shape[0]):
        for j in range(cm.shape[1]):
            ax.text(j, i, cm[i, j], ha="center", va="center", color="black")
    ax.set_xticks(range(len(id2label)))
    ax.set_xticklabels([id2label[i] for i in range(len(id2label))])
    ax.set_yticks(range(len(id2label)))
    ax.set_yticklabels([id2label[i] for i in range(len(id2label))])
ax.set_xlabel("Predicted")
ax.set_ylabel("True")
ax.set_title("Validation Confusion Matrix")
plt.tight_layout()

ModuleNotFoundError: No module named 'matplotlib'

## Results Summary

- Fine-tuning now runs up to **5 epochs** with cosine LR decay, warmup, and early stopping (patience = 2) to converge more reliably; gradient accumulation keeps GPU memory in check.
- Applied both **class-weighted loss** and a **label-aware weighted sampler**, improving exposure to minority classes while still leveraging the full dataset.
- Training uses mixed precision, TF32 matrix math on NVIDIA Ada GPUs, and optional `torch.compile` for extra throughput (works well on RTX 4060).
- Validation logging now surfaces **per-class** and **per-language macro F1**, plus confusion-matrix plots for deeper error analysis.
- The best checkpoint and consolidated training history persist to `xlmr_finetune_best.pt` for easy reuse.

> **Next improvement targets toward ~90% accuracy**: run the notebook for the full dataset (remove the sample cap), expand to 8–10 epochs if validation keeps improving, and experiment with longer sequences (e.g., 192 tokens) once GPU headroom allows. You can further try fp16 LoRA adapters or task-specific data augmentation for additional gains.